In [1]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/nico/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/nico/tugas4/
print("Berhasil diunggah ke HDFS: /user/nico/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/nico/tugas4/transaksi_september_2026.csv


In [3]:
#membaca dan eksplorasi awal
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, avg, when

# Inisialisasi SparkSession
spark = SparkSession.builder.appName("Tugas4_23_Dimas").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Membaca dataset langsung dari HDFS
hdfs_path = "hdfs://localhost:9000/user/nico/tugas4/transaksi_september_2026.csv"
df_tugas = spark.read.csv(hdfs_path, header=True, inferSchema=True)

df_tugas.printSchema()
print("Jumlah baris:", df_tugas.count())
df_tugas.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|        

In [4]:
#menangani data kosong
# Menangani missing value dengan mengisi nilai 3.0 menggunakan df.na.fill
df_clean = df_tugas.na.fill({"rating": 3.0})
print("Missing value berhasil ditangani.")

Missing value berhasil ditangani.


In [5]:
#transformasi data
df_transformed = df_clean.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))
df_transformed = df_transformed.withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil")
)
df_transformed.select("order_id", "total_pendapatan", "tier_transaksi").show(5)

+--------+----------------+--------------+
|order_id|total_pendapatan|tier_transaksi|
+--------+----------------+--------------+
|ORD-3000|          270000|         Kecil|
|ORD-3001|          600000|         Besar|
|ORD-3002|          480000|         Kecil|
|ORD-3003|         2100000|         Besar|
|ORD-3004|          600000|         Besar|
+--------+----------------+--------------+
only showing top 5 rows



In [6]:
#analisi data dengan GroupBY
#kategori dengan pendapatn tertinggi
df_transformed.groupBy("kategori") \
    .agg(spark_sum("total_pendapatan").alias("sum_pendapatan")) \
    .orderBy(col("sum_pendapatan").desc()) \
    .show(1, truncate=False)

+------------+--------------+
|kategori    |sum_pendapatan|
+------------+--------------+
|Rumah Tangga|138665000     |
+------------+--------------+
only showing top 1 row



In [7]:
#Kota dengan jumlah transaksi tier "Besar" terbanyak
df_transformed.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .count() \
    .withColumnRenamed("count", "jumlah_transaksi_besar") \
    .orderBy(col("jumlah_transaksi_besar").desc()) \
    .show(1, truncate=False)

+----+----------------------+
|kota|jumlah_transaksi_besar|
+----+----------------------+
|Solo|92                    |
+----+----------------------+
only showing top 1 row



In [8]:
#Rata-rata rating untuk masing-masing metode_pembayaran
df_transformed.groupBy("metode_pembayaran") \
    .agg(avg("rating").alias("rata_rata_rating")) \
    .orderBy(col("rata_rata_rating").desc()) \
    .show(truncate=False)

+-----------------+-----------------+
|metode_pembayaran|rata_rata_rating |
+-----------------+-----------------+
|COD              |3.948207171314741|
|Transfer Bank    |3.932806324110672|
|E-Wallet         |3.904            |
|Kartu Kredit     |3.861788617886179|
+-----------------+-----------------+



In [9]:
#Menyimpan Hasil ke HDFS
output_hdfs_path = "hdfs://localhost:9000/user/nico/tugas4/hasil_transaksi_september_processed"

df_transformed.write.mode("overwrite").option("header", "true").csv(output_hdfs_path)
print("Hasil berhasil disimpan ke HDFS.")

Hasil berhasil disimpan ke HDFS.
